# Exercise 9.2 — SHAP values: global and local explanations

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day9/day6-02-shap.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

Exercise 9.1 ranked the variables for the forest as a whole. It did not say
which way a variable pushes the risk, and it said nothing about any one
patient. **SHAP values** do both. For every patient and every variable, a SHAP
value says how much that variable moved this patient's predicted probability
away from the average prediction.

**Goal.** In this exercise you will:

- compute SHAP values for the test patients and check that they add up to the
  prediction;
- read two global plots: the mean absolute SHAP bar plot and the beeswarm;
- read the waterfall plot for one patient.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

The same data ([`data/diabetes_clean.csv`](data/diabetes_clean.csv)), outcome
and five predictors as Exercise 9.1.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

As always, 80% of the people are used for training and 20% are held back as
the test set.

In [ ]:
train, test = train_test_split(diabetes, test_size=0.2, random_state=42)

The same random forest as Exercise 9.1, fitted on the training rows and
evaluated once on the test rows.

In [ ]:
rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf = rf.fit(train[predictors], train["diabetes"])

test_prob = rf.predict_proba(test[predictors])[:, 1]    # probability of diabetes

print("test AUC:", round(roc_auc_score(test["diabetes"], test_prob), 3))

The forest reaches a test AUC of 0.794.

------------------------------------------------------------------------

### Part 1 — SHAP values for the test patients

Start from the **baseline**: the average prediction. A patient's SHAP values
share out the gap between the baseline and that patient's prediction, one
piece per variable. A positive value pushes the risk **up**; a negative value
pushes it **down**. Added together, they give the prediction exactly:

**baseline + the SHAP values of all variables = the prediction.**

The **shap** package's `TreeExplainer` uses the structure of the trees to
compute exact SHAP values in under a second. We explain the 150 test patients
and keep the values for class 1, diabetes.

In [ ]:
explainer = shap.TreeExplainer(rf)
explanation = explainer(test[predictors])

shap_pos = explanation[:, :, 1]            # SHAP values for class 1 (diabetes)

print("shape:   ", shap_pos.values.shape)  # one row per patient, one column per predictor
print("baseline:", round(shap_pos.base_values[0], 3))   # the average prediction

The baseline is 0.359. Now check that it adds up for the first test patient.

In [ ]:
# Replace every ______ below, then run this cell.
patient_shap = shap_pos.values[0]
print(np.round(patient_shap, 3))

print(shap_pos.base_values[0] + patient_shap.______())
print(test_prob[0])

> **Stuck?**
>
> You want to add up the five SHAP values in `patient_shap`. NumPy arrays have a
> `.sum()` method.

------------------------------------------------------------------------

### Part 2 — The global view: bar plot and beeswarm

We now have 150 × 5 SHAP values. The **mean absolute SHAP value** of a
variable says how far, on average, it moves a patient's prediction, up or
down. It is a variable importance, in units of probability.

In [ ]:
shap_importance = pd.Series(np.abs(shap_pos.values).mean(axis=0), index=predictors)

shap_importance.sort_values(ascending=False).round(3)

Glucose moves a prediction by 0.159 on average. The other four variables move
it by between 0.034 and 0.066.

The bar plot draws the same numbers:

In [ ]:
shap.plots.bar(shap_pos, show=False)
plt.tight_layout()
plt.show()

The **beeswarm** shows every patient. Each dot is one patient's SHAP value for
one variable. The colour is the patient's value of that variable, blue low and
red high.

In [ ]:
shap.plots.beeswarm(shap_pos, show=False)
plt.tight_layout()
plt.show()

**Question!** In the glucose row, where are the patients with high glucose,
and where are those with low glucose?

High glucose sits on the right: it pushes the risk up, by as much as 0.480.
Low glucose sits on the left and pushes it down, by as much as 0.313. The bar
plot gives only the size of each effect. The beeswarm also gives its direction,
patient by patient.

------------------------------------------------------------------------

### Part 3 — One patient: the waterfall plot

A **local** explanation answers: *why did the model give this patient this
risk?* Take the first test patient again:

In [ ]:
patient = test.iloc[[0]]                   # double brackets keep it a one-row table
print(patient)
print("predicted probability:", round(test_prob[0], 3))

She is 35, with 6 pregnancies, a BMI of 28.8 and a glucose of 115 mg/dL, and
she does **not** have diabetes. The waterfall plot starts at the baseline at
the bottom and adds one SHAP value at a time, up to her prediction at the top.

In [ ]:
shap.plots.waterfall(shap_pos[0], show=False)
plt.tight_layout()
plt.show()

Read in words: the average patient has a predicted risk of 0.359. Her ordinary
glucose takes 0.139 off. Her age, BMI and pregnancies each add a little (0.032
or less), and her pedigree function takes off 0.021. Her predicted risk is
0.276: below 0.5, so she is correctly predicted as `neg`.

**Your turn.** Draw the waterfall for the test patient the model thinks is at
**highest** risk.

In [ ]:
# Replace every ______ below, then run this cell.
highest = test_prob.______()                # the position of the highest probability
print(test.iloc[[highest]])

shap.plots.waterfall(shap_pos[______], show=False)
plt.tight_layout()
plt.show()

> **Stuck?**
>
> NumPy arrays have `.argmax()`, the position of the largest value. Then use that
> position to pick the patient out of `shap_pos`.

> **Key points**
>
> - A SHAP value is one variable's push on one patient's prediction, up or down from the baseline (the average prediction).
> - Baseline + the SHAP values of all variables = the prediction, exactly.
> - Mean absolute SHAP ranks the variables; the beeswarm also shows the direction of each effect.
> - The waterfall plot explains one patient's prediction. It says what the model did, not whether the model was right.